<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/4_ALM_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Asset Lifecycle Management - Evaluation & Quality Assurance

> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

> **Prerequisites:** none in Colab — the setup cell prepares everything. `nat eval` runs the workflow in-process (no server needed).

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand why evaluation matters** for agentic AI systems
2. **Use NAT's eval harness** to test agent performance
3. **Interpret evaluation metrics** (text matching, plot quality)
4. **Create custom evaluation datasets** for your domain
5. **Analyze failure cases** and identify improvement areas
6. **Compare model performance** across different configurations

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

In [ ]:
from nat_client import NATClient, link
client = NATClient()

> **Reset codebase:** If you need to restore all source files and config to their original state (e.g. after testing a solution), run the cell below.

In [ ]:
client.reset()

---
# Section 4.1: Motivation for Evaluation

## Why Evaluation is Critical

### Challenges with Agentic AI

Unlike traditional ML models with clear accuracy metrics, agentic AI systems are complex:

- **Non-deterministic**: Same query can produce different responses
- **Multi-dimensional**: Text quality, tool selection, reasoning correctness
- **Context-dependent**: "Good" responses vary by use case
- **Tool-augmented**: Success depends on correct tool usage
- **Multi-step**: Errors can compound across reasoning chains

### What Makes a "Good" Agent Response?

1. **Factual Correctness**: Returns accurate information
2. **Tool Selection**: Uses appropriate tools for the task
3. **Reasoning Quality**: Logical step-by-step approach
4. **Completeness**: Addresses all parts of the query
5. **Efficiency**: Minimal unnecessary steps
6. **Output Quality**: Correct data format, valid visualizations

### Production Deployment Gates

Before deploying to production, you need to validate:

```
Development → Evaluation → Staging → Production
                   ↓
            Must Pass:
            - Accuracy > 85%
            - No critical failures
            - Latency < 15s (p95)
            - Cost within budget
```

## NAT's Evaluation Framework

NAT provides:

- **Eval Harness**: Automated testing framework
- **Multimodal Evaluator**: Judges both text and visualizations
- **Custom Metrics**: Extensible evaluation logic
- **Batch Processing**: Test multiple queries efficiently
- **Result Analysis**: Structured output for analysis

---
# Section 4.2: Understanding NAT's Evaluator

## The Multimodal LLM Judge

NAT uses a **multimodal LLM** (supports text + images) to evaluate responses:

### Evaluation Process:

```
1. Run Query → Get Agent Response (text + plots)
2. Load Reference Answer from eval dataset
3. Send to Judge LLM:
   - Question
   - Reference Answer
   - Agent Response (text)
   - Generated Plots (images)
4. Judge Returns Score: 0.0, 0.5, or 1.0
```

## Evaluation Dataset Structure

From `eval_set_lab.json`:

```json
[
    {
      "id": "1",
      "question": "What is the ground truth remaining useful life (RUL) of unit_number 59 in dataset FD001",
      "answer": "114 ",
      "type": "text",
      "category": "retrieval",
      "subcategory": "easy",
      "original_id": "1",
      "source": "eval_set"
    },
    {
      "id": "2",
      "question": "In dataset train_FD004, plot sensor_measurement1 vs time_in_cycles for unit_number 107",
      "answer": "Line chart showing sensor_measurement1 values on y-axis ranging from 445.00 to 518.67 plotted against time_in_cycles in x-axisfor unit 107 in dataset FD004.",
      "type": "text_plus_plot",
      "category": "visualization",
      "subcategory": "easy",
      "original_id": "1",
      "source": "eval_set"
    }
]
```

### Explaination of the dataset:

- **id**: Unique identifier
- **question**: Query to test
- **answer**: Expected response (reference)
- **type**: `text`, `text_plus_plot`, or `code`
- **category**: `retrieval`, `visualization`, `analysis`, `prediction`
- **subcategory**: `easy`, `medium`, `hard`

## Scoring Rubric

### Text Evaluation:
- **1.0**: Fully correct, matches reference semantically
- **0.5**: Partially correct, missing some elements
- **0.0**: Incorrect or irrelevant

### Plot Evaluation:
- **1.0**: Shows all major elements from reference
- **0.5**: Shows some elements, missing significant aspects
- **0.0**: Doesn't match reference description

## Configuration (from config-reasoning.yml)

```yaml
eval:
  general:
    output:
      dir: "eval_output"
      cleanup: true
    dataset:
      _type: json
      file_path: "eval_set_lab.json"
    query_delay: 10  # seconds between queries
    max_concurrent: 1
  
  evaluators:
    multimodal_eval:
      _type: multimodal_llm_judge_evaluator
      llm_name: multimodal_judging_llm
```

---
# Section 4.3: Hands-on Evaluation Exercises

## Exercise 4.1: Run Basic Eval Harness

### Step 1: Examine the Evaluation Dataset

In [ ]:
import json
import pandas as pd

# Load and display the eval dataset
eval_file = f"{ALM_HOME}/eval_set_lab.json"

with open(eval_file, 'r') as f:
    eval_data = json.load(f)

# Convert to DataFrame for better viewing
df = pd.DataFrame(eval_data)
print(f"Total evaluation queries: {len(df)}\n")
print("Sample queries:")
print(df[['id', 'category', 'subcategory', 'question']].head(10))

# Count by category
print("\nQueries by category:")
print(df['category'].value_counts())

print("\nQueries by difficulty:")
print(df['subcategory'].value_counts())

### Step 2: Run the Evaluation

In the lab you would run this in a terminal:

```bash
source /content/alm-venv/bin/activate
nat eval --config_file configs/config-reasoning.yml --dataset /content/NATcolab/eval_set_lab.json
```

**Important:** Make sure you pass the --dataset flag so that it uses the local file!

`client.nat(...)` runs the same CLI from the notebook and streams its output (≈3–6 minutes: each query runs through the full agent, then the multimodal judge scores it). This will:
1. Load queries from the eval dataset
2. Send each query to the agent
3. Collect responses (text + plots)
4. Send to multimodal judge for scoring
5. Save results to `eval_output/`

In [ ]:
rc = client.nat("eval", "--config_file", "configs/config-reasoning.yml", "--dataset", f"{ALM_HOME}/eval_set_lab.json")
print("exit code:", rc)

### Step 3: Analyze Results

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Load evaluation results
eval_output_dir = f"{ALM_HOME}/eval_output"
results_file = Path(eval_output_dir) / "multimodal_eval_output.json"

if results_file.exists():
    with open(results_file, 'r') as f:
        data = json.load(f)
    
    # Extract the eval_output_items list
    results_df = pd.DataFrame(data['eval_output_items'])
    
    print("=== Evaluation Results ===")
    print(f"Total queries: {len(results_df)}")
    print(f"Average score (from file): {data['average_score']:.2f}")
    print(f"Average score (calculated): {results_df['score'].mean():.2f}")
    print(f"Median score: {results_df['score'].median():.2f}")
    print(f"\nScore distribution:")
    print(results_df['score'].describe())
    
    # Show sample results
    print(f"\n=== Sample Results ===")
    print(results_df[['id', 'score']].head(10).to_string())
else:
    print("No results file found. Run the eval first!")

### Step 4: Investigate Failure Cases

For each failed query, analyze:
1. What did the agent return?
2. What was expected?
3. Why did it fail? (wrong tool, incorrect data, reasoning error)
4. How can we fix it? (config change, prompt update, tool enhancement)

In [ ]:
# Deep dive into a specific failed query
query_id = "1"  # Change this to investigate specific queries (ids are strings in the dataset)

if results_file.exists():
    with open(results_file, 'r') as f:
        data = json.load(f)
    
    # Find the query from eval_output_items
    query_result = next((r for r in data['eval_output_items'] if str(r['id']) == str(query_id)), None)
    
    if query_result:
        print(f"Query ID: {query_result['id']}")
        print(f"Score: {query_result['score']}")
        print(f"\nReasoning: {query_result['reasoning']}")
        print(f"\nNote: Use workflow_output.json to see full query details including question, expected answer, and agent response.")
    else:
        print(f"Query {query_id} not found")

In [ ]:
# Full details (question, expected answer, generated answer) live in workflow_output.json
import json
wf = Path(eval_output_dir) / "workflow_output.json"
if wf.exists():
    for item in json.load(open(wf)):
        print(f"--- id={item.get('id')}\nQ: {item.get('question')}\nExpected: {item.get('answer')}\n"
              f"Agent: {str(item.get('generated_answer'))[:800]}\n")

---
## Exercise 4.2: Create Custom Eval Dataset

### Objective

Create your own evaluation dataset with 3-5 queries covering different categories and difficulties.

### Step 1: Design Your Queries

Consider:
- **Easy**: Simple data retrieval
- **Medium**: Retrieval + visualization or simple analysis
- **Hard**: Multi-step analysis, anomaly detection, recommendations

### Step 2: Create the Dataset File

In [ ]:
import json

# Create custom evaluation dataset
custom_eval = [
    {
        "id": "custom_1",
        "question": "How many units are in the FD002 training dataset?",
        "answer": "260",
        "type": "text",
        "category": "retrieval",
        "subcategory": "easy"
    },
    {
        "id": "custom_2",
        "question": "Plot the operational_setting_1 values for unit 100 in FD002",
        "answer": "Line chart showing operational_setting_1 vs time_in_cycles for unit 100",
        "type": "text_plus_plot",
        "category": "visualization",
        "subcategory": "medium"
    },
    # Add your own queries here!
]

# Save to file
custom_file = f"{ALM_HOME}/eval_set_custom.json"
with open(custom_file, 'w') as f:
    json.dump(custom_eval, f, indent=2)

print(f"✅ Created custom evaluation dataset with {len(custom_eval)} queries")
print(f"Saved to: {custom_file}")

### Step 3: Run Evaluation on Custom Dataset

Update the config to use your custom dataset:

```yaml
eval:
  general:
    dataset:
      _type: json
      file_path: "eval_set_custom.json"  # ← Change this
```

Then run:
```bash
nat eval --config_file configs/config-reasoning.yml
```

or run:
```bash
nat eval --config_file configs/config-reasoning.yml --dataset /content/NATcolab/eval_set_custom.json
```

### Step 4: Compare Results

Compare performance on:
- Original eval set vs your custom set
- Different categories
- Different difficulty levels

Document insights:
- Which types of queries does the agent handle well?
- Which types need improvement?
- Are there systematic failures?

In [ ]:
rc = client.nat("eval", "--config_file", "configs/config-reasoning.yml", "--dataset", f"{ALM_HOME}/eval_set_custom.json")
print("exit code:", rc)
data = json.load(open(Path(eval_output_dir) / "multimodal_eval_output.json"))
print(f"Custom set average score: {data['average_score']:.2f}")
for r in data["eval_output_items"]:
    print(r["id"], r["score"], str(r.get("reasoning"))[:200])

---
# Key Takeaways

In this notebook, you learned:

1. ✅ Why evaluation is critical for production deployment
2. ✅ How NAT's multimodal evaluator works
3. ✅ How to run the eval harness
4. ✅ How to interpret evaluation metrics
5. ✅ How to create custom evaluation datasets
6. ✅ How to analyze failure cases
7. ✅ How to iterate and improve agent performance

## Evaluation Best Practices

- **Test early and often**: Don't wait until production
- **Cover edge cases**: Include difficult and ambiguous queries
- **Track over time**: Monitor regressions when making changes
- **Diverse dataset**: Mix categories and difficulties
- **Real user queries**: Include actual user inputs
- **Continuous improvement**: Use eval results to guide development

## What's Next?

In **Notebook 5**, you will:
- Learn about time-series analysis for asset management
- Understand foundation models for time-series (NV-Tesseract, MOMENT)
- Run anomaly detection on turbofan engine sensor data
- Compare anomaly patterns across engines with different health profiles

**[Continue to Notebook 5 - Time-Series Modeling →](5_ALM_AnomalyDetection.ipynb)**